# Learning objectives

- Define a local function as a strict Prompt Agent tool.
- Combine a local function tool with Foundry Web Search.
- Return local function output to the agent and inspect the completed response.

# Define the tools

`get_weather` is a deliberately simulated local data source.  
Foundry stores its schema in the agent definition, but the notebook executes the Python function and returns its output.   
`WebSearchTool` runs in Foundry and is intended for the current public-web part of the request.

In [ ]:
# Import configuration helpers, Foundry SDK types, and response-tool output types.
import json
import os
from random import choice, randint

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    FunctionTool,
    PromptAgentDefinition,
    Tool,
    WebSearchTool,
)
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai.types.responses.response_input_param import (
    FunctionCallOutput,
    ResponseInputParam,
)
from workshop_core.config import WorkshopConfig
from workshop_core.naming import build_resource_name, sdk_participant_id

# Load configuration and create a participant-scoped agent name.
load_dotenv()
config = WorkshopConfig.load()
sdk_participant = sdk_participant_id(config.participant_id)
agent_name = build_resource_name(sdk_participant, "multi-tools", "agent")

# Connect with the Azure identity established by az login.
credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential,
)
openai = project.get_openai_client()

print(f"SDK agent name: {agent_name}")

In [ ]:
# This function simulates a local line-of-business weather source.
def get_weather(location: str) -> str:
    conditions = ["sunny", "cloudy", "rainy", "windy"]
    condition = choice(conditions)
    temperature_celsius = randint(12, 28)
    return (
        f"Simulated weather for {location}: {condition}, "
        f"{temperature_celsius} degrees Celsius."
    )


# Describe the callable function for the Prompt Agent.
weather_tool = FunctionTool(
    name="get_weather",
    description="Return simulated weather for a requested city or region.",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city or region to get simulated weather for.",
            }
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    strict=True,
)

# Web Search is a managed Foundry tool for current public-web information.
web_search_tool = WebSearchTool()
tools: list[Tool] = [weather_tool, web_search_tool]

# Create and run the multi-tool agent

The request needs two sources: a simulated weather result from the local `get_weather` function and a current Foundry announcement from Web Search.

The `instructions` and `prompt` describe the intended routing: weather belongs to `get_weather`, while the announcement belongs to Web Search.  
They guide the model, but do not guarantee which tool it selects first.   
`tool_choice={"type": "function", "name": "get_weather"}` makes the first stage deterministic by requiring the named local function.

In [ ]:
# Tell the agent exactly which source owns each part of the request.
instructions = (
    "You are a helpful assistant. For every weather question, call get_weather and "
    "use only its returned simulated weather data. Never use Web Search for weather, "
    "forecasts, conditions, or weather advisories. Clearly label the weather result as "
    "simulated. Use Web Search only for a separately requested current public-web "
    "topic, and include source links when available."
)
prompt = (
    "First, call get_weather for a simulated weather report for Tel Aviv. Do not use "
    "Web Search for any weather information. Separately, use Web Search only to "
    "identify a current Microsoft Foundry announcement and give its URL."
)

# Persist an agent version that has both tool definitions.
multi_tool_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=config.model_deployment_name,
        instructions=instructions,
        tools=tools,
    ),
)
print(
    f"Created agent {multi_tool_agent.name} "
    f"(version {multi_tool_agent.version})."
)

# Start a conversation so the function output can continue this interaction.
conversation = openai.conversations.create()
print(f"Conversation ID for Foundry Traces: {conversation.id}")
initial_response = openai.responses.create(
    conversation=conversation.id,
    input=prompt,
    tool_choice={"type": "function", "name": "get_weather"},
    extra_body={
        "agent_reference": {
            "name": multi_tool_agent.name,
            "type": "agent_reference",
        }
    },
)

This workflow requires two responses because Foundry can request a local function but cannot run notebook Python.  
The first response contains the `function_call`.  
The notebook then runs `get_weather` and submits a `function_call_output` in the same conversation.   
The second response can use that returned weather data, use Web Search for the separate announcement request, and produce the combined answer.

After the next cell prints the combined answer, open the Foundry portal and go to **Agents**. Locate the SDK agent using the name printed by the notebook and inspect both tool definitions. Open **Traces** and use the printed conversation ID to follow the SDK-created function call, local function output, and final response.

In [ ]:
# Execute only the local function calls requested by the agent.
function_outputs: ResponseInputParam = []
for item in initial_response.output:
    if item.type != "function_call" or item.name != "get_weather":
        continue

    arguments = json.loads(item.arguments)
    weather = get_weather(**arguments)
    print(f"Local get_weather invocation: {arguments}")
    print(weather)
    function_outputs.append(
        FunctionCallOutput(
            type="function_call_output",
            call_id=item.call_id,
            output=json.dumps({"weather": weather}),
        )
    )

if not function_outputs:
    raise RuntimeError(
        "The agent did not request get_weather. Re-run with the supplied prompt."
    )

# Return the local function result so the agent can complete the response.
final_response = openai.responses.create(
    conversation=conversation.id,
    input=function_outputs,
    extra_body={
        "agent_reference": {
            "name": multi_tool_agent.name,
            "type": "agent_reference",
        }
    },
)

print("\nFinal response:\n", final_response.output_text)